# Preprocessing

# Setup

In [ ]:
from thesis_project import (
    config,
)

config.default()

TICKER: config.FutTicker = "fbtp"

# Lobs directories
LOB_DIR = config.DATA_RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "fut"
LOB_CTD_DIR = LOB_DIR / "ctd"

# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.RES_FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_EXP_DIR = config.RES_EXP_DIR / TICKER / NOTEBOOK
    OUT_EXP_DIR.mkdir(parents=True, exist_ok=True)

# Common

In [ ]:
from typing import get_args

from thesis_project import dataset, plots

In [ ]:
collections: dict[config.AssetRole, dataset.lob.LobReportCollector] = {}
for role in get_args(config.AssetRole):
    collections[role] = dataset.lob.load_lob_reports(TICKER, role)

# Main

## NaNs

In [ ]:
key = "n_rows_any_nan_perc"
plots.lob_report.nans_hist_per_ticker(collections, key)

## Timestamps

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def foo(collection: dataset.lob.LobReportCollector):
    opening_ts: list[pd.Timestamp] = []
    closing_ts: list[pd.Timestamp] = []
    for report in collection.reports.values():
        opening_ts.append(report.integrity["timestamps"]["levels"][1]["min_valid_idx"])
        closing_ts.append(report.integrity["timestamps"]["levels"][1]["max_valid_idx"])

    opening_sec = [
        ts.hour * 60**2 + ts.minute * 60 + ts.second for ts in opening_ts if ts is not None
    ]
    closing_sec = [
        ts.hour * 60**2 + ts.minute * 60 + ts.second for ts in closing_ts if ts is not None
    ]

    asset = config.ASSET_BY_TICKER_ROLE[(collection.ticker, collection.role)]

    opening = asset.market.opening_time
    opening = opening.hour * 60**2 + opening.minute * 60 + opening.second
    closing = asset.market.closing_time
    closing = closing.hour * 60**2 + closing.minute * 60 + closing.second
    bin_width = 10 * 60
    bins = range(opening, closing + bin_width, bin_width)
    tick_width = 60**2
    xticks = range(opening, closing + tick_width, tick_width)

    xlabels = [
        f"{int(t // 60**2):02d}:" f"{int((t % 60**2) // 60):02d}:" f"{int(t % 60):02d}"
        for t in xticks
    ]

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.hist(
        opening_sec,
        bins=bins,
        label="opening",
        edgecolor="black",
        linewidth=0.5,
    )

    ax.hist(
        closing_sec,
        bins=bins,
        label="closing",
        edgecolor="black",
        linewidth=0.5,
    )

    ax.set_xlabel("Time")
    ax.set_xticks(xticks)
    ax.set_xticks(bins, minor=True)
    ax.set_xticklabels(xlabels)

    ax.set_ylabel("Count")
    ax.set_yscale("log")

    ax.set_title("Timestamps distribution")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

In [ ]:
for role in get_args(config.AssetRole):
    foo(collections[role])

## Gaps